# Phase 2.2 Monte Carlo — Visualisation

Three plots for the slide deck:
1. **Loss distribution overlay** — historical vs MC, full + tail zoom
2. **Q-Q plots** — simulated vs historical for SPY, AGG, GLD, TLT
3. **CVaR comparison bar chart** — β = 0.95 / 0.975 / 0.99

Inputs expected (upload to `/content/` or use the `files.upload()` cell):
- `scenario_returns_matrix.csv` — Sijing's §2.3 historical scenarios
- `scenario_returns_matrix_mc.csv` *or* `scenario_returns_matrix_mc.npz` — §2.2 MC scenarios

## 0. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['font.size'] = 10

# Colors used throughout
C_HIST = '#4C72B0'   # blue  = historical
C_MC   = '#DD8452'   # orange = Monte Carlo
C_TAIL = '#C44E52'   # red   = tail / VaR / CVaR markers

## 1. Load the data

If running fresh in Colab and you haven't uploaded yet, uncomment the `files.upload()` cell. Otherwise the next cell will look in `/content/` and the current directory.

In [ ]:
# Uncomment to upload via the Colab widget:
# from google.colab import files
# files.upload()

In [ ]:
def find(name, dirs=('/content', '.', './Phase2/2.3', './Phase2/2.2')):
    for d in dirs:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(name)

# Historical (§2.3)
hist_path = find('scenario_returns_matrix.csv')
R_hist_df = pd.read_csv(hist_path, index_col=0, parse_dates=True)
R_hist = R_hist_df.values
print(f'Historical: {hist_path}\n  shape = {R_hist.shape}')

# MC (§2.2) — prefer .npz for speed if both present
try:
    mc_path = find('scenario_returns_matrix_mc.npz')
    archive = np.load(mc_path, allow_pickle=True)
    R_mc = archive['scenarios']
    cols = [str(c) for c in archive['columns']]
    R_mc_df = pd.DataFrame(R_mc, columns=cols)
    print(f'MC:         {mc_path} (npz)\n  shape = {R_mc.shape}')
except FileNotFoundError:
    mc_path = find('scenario_returns_matrix_mc.csv')
    R_mc_df = pd.read_csv(mc_path, index_col=0)
    R_mc = R_mc_df.values
    print(f'MC:         {mc_path} (csv)\n  shape = {R_mc.shape}')

# Sanity check that columns line up.
assert list(R_hist_df.columns) == list(R_mc_df.columns), 'column mismatch'
ASSETS = list(R_hist_df.columns)
N = len(ASSETS)
print(f'\n{N} assets: {ASSETS}')

## 2. Equal-weight portfolio losses + CVaR helpers

In [ ]:
# Equal weights — placeholder until §3.1 produces optimal w.
w = np.full(N, 1.0 / N)

L_hist = -(R_hist @ w)   # losses (positive = loss)
L_mc   = -(R_mc   @ w)

def cvar_empirical(losses, beta):
    losses = np.asarray(losses).ravel()
    S = losses.size
    k = max(int(np.ceil((1.0 - beta) * S - 1e-9)), 1)
    return float(np.partition(losses, -k)[-k:].mean())

def cvar_ru(losses, beta):
    """Rockafellar-Uryasev CVaR + VaR, closed-form via order statistic."""
    L = np.sort(np.asarray(losses).ravel())
    S = L.size
    idx = min(max(int(np.ceil(beta * S)) - 1, 0), S - 1)
    alpha = float(L[idx])
    excess = np.maximum(losses - alpha, 0.0).sum()
    return alpha + excess / ((1.0 - beta) * S), alpha

# Quick numbers we'll use as annotations
betas = (0.95, 0.975, 0.99)
summary = []
for b in betas:
    ch, ah = cvar_ru(L_hist, b)
    cm, am = cvar_ru(L_mc,   b)
    summary.append({
        'β': b,
        'tail_hist': max(int(np.ceil((1-b)*L_hist.size - 1e-9)), 1),
        'tail_mc':   max(int(np.ceil((1-b)*L_mc.size   - 1e-9)), 1),
        'VaR_hist':  ah,  'CVaR_hist': ch,
        'VaR_mc':    am,  'CVaR_mc':   cm,
    })
summary_df = pd.DataFrame(summary)
summary_df

## 3. Plot 1 — Loss distribution overlay (full + tail zoom)

Left panel: full distribution. The MC histogram fills in the gaps Sijing's 204 historical observations leave behind.

Right panel: zoomed to the worst 5% (β = 0.95 left tail). Vertical lines = empirical VaR / CVaR for each. The whole point of MC: smooth, well-resolved tail instead of 11 lonely observations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# ----- Left: full distribution -----
ax = axes[0]
bins_full = np.linspace(min(L_hist.min(), L_mc.min()),
                        max(L_hist.max(), L_mc.max()), 60)
ax.hist(L_mc,   bins=bins_full, density=True, alpha=0.55, color=C_MC,
        label=f'MC  (S = {L_mc.size:,})')
ax.hist(L_hist, bins=bins_full, density=True, alpha=0.65, color=C_HIST,
        label=f'Historical (S = {L_hist.size})')
ax.set_xlabel('Equal-weight portfolio loss')
ax.set_ylabel('Density')
ax.set_title('Loss distribution: historical vs Monte Carlo')
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.axvline(0, color='k', lw=0.5, alpha=0.4)
ax.legend(frameon=False, loc='upper left')

# ----- Right: tail zoom (worst 5%) -----
ax = axes[1]
var_hist = np.quantile(L_hist, 0.95)
tail_lo  = np.quantile(np.concatenate([L_hist, L_mc]), 0.92)  # show a bit beyond 95
tail_hi  = max(L_hist.max(), np.quantile(L_mc, 0.9999))
bins_tail = np.linspace(tail_lo, tail_hi, 40)
ax.hist(L_mc,   bins=bins_tail, density=True, alpha=0.55, color=C_MC,
        label='MC tail (β ≥ 0.95)')
ax.hist(L_hist, bins=bins_tail, density=True, alpha=0.65, color=C_HIST,
        label='Historical tail')

# VaR / CVaR markers from §2
row = summary_df[summary_df['β'] == 0.95].iloc[0]
ax.axvline(row['VaR_hist'],  color=C_HIST, ls='--', lw=1.4,
           label=f"Hist VaR={row['VaR_hist']:.2%}")
ax.axvline(row['CVaR_hist'], color=C_HIST, ls=':',  lw=1.4,
           label=f"Hist CVaR={row['CVaR_hist']:.2%}")
ax.axvline(row['VaR_mc'],    color=C_MC,   ls='--', lw=1.4,
           label=f"MC VaR={row['VaR_mc']:.2%}")
ax.axvline(row['CVaR_mc'],   color=C_MC,   ls=':',  lw=1.4,
           label=f"MC CVaR={row['CVaR_mc']:.2%}")
ax.set_xlabel('Loss (left tail, worst ~8%)')
ax.set_ylabel('Density')
ax.set_title('Tail zoom: β = 0.95 VaR / CVaR')
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend(frameon=False, fontsize=8.5, loc='upper right')

plt.tight_layout()
fig.savefig('plot1_loss_distribution.png', bbox_inches='tight')
plt.show()

## 4. Plot 2 — Q-Q plots (simulated vs historical)

For each ticker, sort the 204 historical returns and the MC returns, then plot the matching empirical quantiles against each other. Points on the y = x line mean the MC marginal matches the historical marginal. Departures in the bottom-left corner = tail mismatch.

In [ ]:
PICKS = ['SPY', 'AGG', 'GLD', 'TLT']
PICKS = [t for t in PICKS if t in ASSETS]   # safety if names change

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, t in zip(axes.flat, PICKS):
    j = ASSETS.index(t)
    h = np.sort(R_hist[:, j])
    n = h.size
    p = (np.arange(1, n + 1) - 0.5) / n
    s = np.quantile(R_mc[:, j], p)   # MC quantiles at the same probabilities
    ax.scatter(h, s, s=12, alpha=0.7, color=C_HIST)
    lo = min(h.min(), s.min())
    hi = max(h.max(), s.max())
    ax.plot([lo, hi], [lo, hi], color=C_TAIL, lw=1.2, ls='--', label='y = x')
    ax.set_xlabel('Historical quantile')
    ax.set_ylabel('MC quantile')
    ax.set_title(f'{t}  —  Q-Q simulated vs historical')
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.legend(frameon=False, loc='upper left', fontsize=9)
    ax.grid(alpha=0.25)

plt.tight_layout()
fig.savefig('plot2_qq_marginals.png', bbox_inches='tight')
plt.show()

## 5. Plot 3 — CVaR comparison bar chart

Side-by-side bars of historical vs MC equal-weight CVaR at β = 0.95 / 0.975 / 0.99, with the effective tail-sample size annotated under each historical bar (the fragility we're fixing).

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))

x = np.arange(len(betas))
width = 0.36

bars_h = ax.bar(x - width/2, summary_df['CVaR_hist'], width,
                color=C_HIST, label=f'Historical (S = {L_hist.size})')
bars_m = ax.bar(x + width/2, summary_df['CVaR_mc'],   width,
                color=C_MC,   label=f'MC (S = {L_mc.size:,})')

# Value labels on top of each bar
for bars, col in [(bars_h, 'CVaR_hist'), (bars_m, 'CVaR_mc')]:
    for bar, v in zip(bars, summary_df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.2%}', ha='center', va='bottom', fontsize=9)

# Tail-size annotations under x labels
labels = []
for _, row in summary_df.iterrows():
    labels.append(f"β = {row['β']:.3f}\nhist tail: {int(row['tail_hist'])} obs\nMC tail: {int(row['tail_mc']):,} obs")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Equal-weight portfolio CVaR (loss)')
ax.set_title('CVaR: historical (S = 204) vs Monte Carlo (S = 50,000)')
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
ax.legend(frameon=False, loc='upper left')
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

# Headroom so labels don't get clipped
ax.set_ylim(0, max(summary_df[['CVaR_hist','CVaR_mc']].max()) * 1.18)

plt.tight_layout()
fig.savefig('plot3_cvar_comparison.png', bbox_inches='tight')
plt.show()

## 6. Done

Three PNGs saved next to this notebook:
- `plot1_loss_distribution.png`
- `plot2_qq_marginals.png`
- `plot3_cvar_comparison.png`

Right-click → download to drop them straight into the slide deck.

In [ ]:
# Optional: bulk-download all three PNGs from Colab
# from google.colab import files
# for fn in ['plot1_loss_distribution.png',
#           'plot2_qq_marginals.png',
#           'plot3_cvar_comparison.png']:
#     files.download(fn)